In [ ]:
import pandas as pd
import os
import string
import re
import nltk



In [ ]:
import pandas as pd
job_dataset = pd.read_csv(r"/home/dell/Desktop/skill_csv/job_descriptions.csv")
display(job_dataset)



In [ ]:
unique_skills = job_dataset['skills'].unique()
print(unique_skills)


In [ ]:
missing_values = job_dataset.isnull().sum()
# display("Missing Values:\n", missing_values)

In [ ]:
job_dataset = job_dataset.dropna()
# display(job_dataset)

In [ ]:
# Example: Convert 'Job Posting Date' to datetime format
job_dataset['Job Posting Date'] = pd.to_datetime(job_dataset['Job Posting Date'])


In [ ]:
# removed latitude and longitude columns,since we have country
job_dataset = job_dataset.drop(['latitude', 'longitude', 'Contact'], axis=1)
# display(job_dataset)

In [ ]:
# covert all data to lower case
job_dataset = job_dataset.applymap(lambda x: x.lower() if isinstance(x, str) else x)
# display(job_dataset)

In [ ]:
def remove_punctuation_from_cell(cell):
    translator = str.maketrans("", "", string.punctuation)
    return str(cell).translate(translator)

for column in job_dataset.columns:
    if  job_dataset[column].dtype == 'object': # Check if the column contains text
        job_dataset[column] = job_dataset[column].apply(remove_punctuation_from_cell)
        
#         display(job_dataset)

In [ ]:
  # adding a standardized format with a hyphen in Salary Range Column
def clean_salary_range(salary_range):
    min_salary = salary_range[:2]
    max_salary = salary_range[3:]
    
    #adding a standardized format with a hyphen
    standardized_salary_range = f"{min_salary}-{max_salary}"
    
    return standardized_salary_range

job_dataset['Salary Range'] = job_dataset['Salary Range'].apply(clean_salary_range)
# display(job_dataset)


In [ ]:
# putting spaces b2n words in Compony profile column
def add_spaces(text):
    clean_text = re.sub(r'(sector|industry|other_keyword)', r'\1 ', text)
    
    return clean_text

job_dataset['Company Profile'] = job_dataset['Company Profile'].apply(add_spaces)

def add_space_before_industry(text):
    return text.replace('industry', ' industry')

job_dataset['Company Profile'] = job_dataset['Company Profile'].apply(add_space_before_industry)
# display(job_dataset)

In [ ]:
# Have created a version of my dataset, of the cleaned data
job_dataset.to_csv('cleaned_data.csv', index=False)

In [ ]:
import pandas as pd
cleaned_dataset = pd.read_csv('/home/dell/Desktop/Skill_code/cleaned_data.csv')
cleaned_dataset.head()

In [ ]:
import pandas as pd
cleaned_dataset = pd.read_csv('/home/dell/Desktop/skill_csv/skill_token.csv')
cleaned_dataset.head()

In [ ]:
from nltk.tokenize import word_tokenize
# Tokenize Qualifications column
cleaned_dataset['Qualifications'] = cleaned_dataset['Qualifications'].apply(lambda x: word_tokenize(str(x)))

# Tokenize Work Type column
cleaned_dataset['Work Type'] = cleaned_dataset['Work Type'].apply(lambda x: word_tokenize(str(x)))

# Tokenize Preference column
cleaned_dataset['Preference'] = cleaned_dataset['Preference'].apply(lambda x: word_tokenize(str(x)))

display(cleaned_dataset)

In [ ]:
from nltk.tokenize import sent_tokenize


# Tokenize company names as single tokens using sentence tokenization
cleaned_dataset['Company'] = cleaned_dataset['Company'].apply(lambda x: sent_tokenize(x))
cleaned_dataset['Role'] = cleaned_dataset['Role'].apply(lambda x: sent_tokenize(x))
cleaned_dataset['Job Title'] = cleaned_dataset['Job Title'].apply(lambda x: sent_tokenize(x))
# Display the tokenized DataFrame
display(cleaned_dataset)


In [ ]:

cleaned_dataset['Company Profile'] = cleaned_dataset['Company Profile'].str.replace('sector', '', regex=False)
display(cleaned_dataset)

In [ ]:
cleaned_dataset.to_csv('dataset_token.csv', index=False)

In [ ]:
import pandas as pd                 
working = pd.read_csv('working_best.csv')
display(working)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans


skills = working['tokenized_skills'].tolist()

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(skills)

# Apply K-means clustering
n_clusters = 6  # Define the number of clusters
kmeans = KMeans(n_clusters=n_clusters)
kmeans.fit(X)

# Predict the cluster for each skill
skill_clusters = kmeans.predict(X)

# Calculate the distribution of samples across clusters
cluster_counts = np.bincount(skill_clusters)
total_samples = len(skills)

# Find the cluster with the highest and lowest demand
highest_demand_cluster = np.argmax(cluster_counts)
lowest_demand_cluster = np.argmin(cluster_counts)

# Prompt user to enter their skills
user_input = input("Enter your skills separated by commas: ")
user_skills = [skill.strip() for skill in user_input.split(',')]

# Transform user skills using the same TF-IDF vectorizer
user_skills_transformed = vectorizer.transform(user_skills)

# Predict the cluster for each user skill
user_skill_clusters = kmeans.predict(user_skills_transformed)

# Determine the demand level of each skill
demand_levels = []
for skill, cluster in zip(user_skills, user_skill_clusters):
    if cluster == highest_demand_cluster:
        demand_levels.append("High demand")
    elif cluster == lowest_demand_cluster:
        demand_levels.append("Low demand")
    else:
        demand_levels.append("Middle demand")

# Display the demand level for each skill
for skill, demand_level in zip(user_skills, demand_levels):
    print(f"Skill: {skill}, Demand Level: {demand_level}")


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd

# Load your DataFrame with the tokenized_skills column
# For example:
# df = pd.read_csv('your_dataset.csv')

# Extract the skills from the tokenized_skills column
skills = working['tokenized_skills'].tolist()

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(skills)

# Apply K-means clustering
n_clusters = 6  # Define the number of clusters
kmeans = KMeans(n_clusters=n_clusters)
kmeans.fit(X)

# Predict the cluster for each skill
skill_clusters = kmeans.predict(X)

# Calculate the distribution of samples across clusters
cluster_counts = np.bincount(skill_clusters)
total_samples = len(skills)

# Find the cluster with the highest and lowest demand
highest_demand_cluster = np.argmax(cluster_counts)
lowest_demand_cluster = np.argmin(cluster_counts)

# Create a DataFrame to store the skills and their demand levels
demand_dataframe = pd.DataFrame(columns=['Skill', 'Demand Level'])
skills_list = []
demand_levels_list = []

# Iterate through each skill and determine its demand level
for skill, cluster in zip(skills, skill_clusters):
    if cluster == highest_demand_cluster:
        demand_level = "High demand"
    elif cluster == lowest_demand_cluster:
        demand_level = "Low demand"
    else:
        demand_level = "Middle demand"
    demand_df = pd.DataFrame({'Skill': skills_list, 'Demand Level': demand_levels_list})
# Display the DataFrame containing skills and their demand levels
print(demand_dataframe)
